In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_cancelamentos_devolucoes_plantas AS
-- ==============================================================================
-- CONSULTA CONSOLIDADA DE DEVOLUÇÕES (RANGE MAXIMIZADO - TODAS AS TOPS)
-- ==============================================================================

-- TOPs de devolução validadas, que geram entrada FÍSICA do produto em estoque:

-- TOPs de Devolução Pessoa Física (a nota fiscal de devolução é emitida pela Belmicro): 2220, 2201, 2234, 2227 (FULL), 2284, 2252, 2285

-- TOPs de Devolução Pessoa Jurídica (a nota fiscal de devolução é emitida pelo revendedor/lojista): 2210, 2215, 2221, 2200, 2233, 2249, 2259, 2230,

-- TOPs de Operação Triangular (Produto vendido para um intermediário, e entregue ao destinatário final pela própria Belmicro): 2290 (PF), 2283 (PJ)

WITH itens_unicos AS (
    SELECT 
        I.NumUnicoNota,
        I.CodProduto,
        MAX(CAST(I.QtdNegociada AS INT)) AS QtdNegociada,
        SUM(CAST(I.ValorTotal AS DECIMAL(18,2))) AS ValorTotal
    FROM gold.sankhya.fato_itens I
    GROUP BY I.NumUnicoNota, I.CodProduto
)
SELECT 
    FI.CodProduto AS SKU,
    PR.DescricaoProduto,
    GP.NomeGrupoPai AS Familia,
    PR.Marca,
    PR.UsadoComo,
    GP.LinhaDeNegocio,
    FO.NumNota,
    FO.NumUnicoNota,
    FO.DataNegociacao,
    FO.TipoMovimento,
    FO.CodTipoOperacao,         
    FO.DescricaoTipoOperacao, 
    FO.StatusNota,
    P.NomeParceiro,
    CASE 
        WHEN P.TipoPessoa = 'J' THEN 'B2B'
        WHEN P.TipoPessoa = 'F' THEN 'B2C'
        ELSE 'OUTROS'
    END AS Segmento,
    V.NomeVendedor,
    FO.NomeGerente,
    
    -- ==============================================================================
    -- LÓGICA DE PLANTAS UNIFICADA (IDÊNTICA À QUERY DE EXPEDIÇÃO OFICIAL)
    -- ==============================================================================
    CASE 
        -- 1. EXCLUSIVIDADE MANAUS/AM (Soberana - Apenas TVs HQ listadas)
        WHEN FI.CodProduto IN (82076, 84047, 84044, 88335, 88334, 88336, 88337, 118198, 118201, 96633, 96635, 96637, 96638) AND PR.Marca = 'HQ' THEN 'MANAUS/AM'
        
        -- 2. HISTÓRICO DE TVs CONTAGEM/SERRA (Transição pós-2025)
        WHEN FI.CodProduto IN (66522, 56549, 66523, 55836, 64018, 66524, 56554, 64019, 71834, 66525, 66526, 71828, 58481, 70119, 63945, 70118, 71215, 68043, 68041, 68040, 71830, 69674, 69669, 69277, 77217, 77736) THEN 
            CASE WHEN YEAR(FO.DataNegociacao) <= 2025 THEN 'CONTAGEM/MG' ELSE 'SERRA/ES' END
            
        -- 3. MONITORES (Regra estrita para impedir vazamento de marcas como Samsung/LG)
        WHEN GP.NomeGrupoPai IN ('MONITORES', 'MONITOR') THEN 
            CASE 
                WHEN FI.CodProduto IN (81829, 81830, 81831, 83115, 83116, 83117, 83118, 84285, 91348, 91346, 91347, 94983, 94988, 94985, 94982, 94986, 94987, 94981, 114447, 114448) AND PR.Marca = 'HQ' THEN 'CONTAGEM/MG' 
                WHEN PR.Marca = 'HQ' THEN 'SERRA/ES'
                WHEN FI.CodProduto IN (21234, 23284, 36027, 49231, 63963, 67997, 76309, 76310, 76311, 76312, 76313, 76314, 76315, 76316, 76317, 77131, 77132, 79314, 79315, 79912) THEN 'SERRA/ES'
                ELSE 'TERCEIRIZADOS' 
            END
            
        -- 4. DESKTOPS, PC, AIO, ALL IN ONE
        WHEN GP.NomeGrupoPai IN ('DESKTOP', 'PC', 'AIO', 'ALL IN ONE') THEN
            CASE 
                WHEN GP.NomeGrupoPai = 'ALL IN ONE' AND PR.Marca NOT IN ('HQ', '3GREEN') THEN 'TERCEIRIZADOS'
                WHEN PR.Marca IN ('HQ', 'CORPC', '3GREEN', 'SKILL', 'EASYPC', 'QUANTUM') THEN 'CONTAGEM/MG'
                ELSE 'TERCEIRIZADOS'
            END
        
        -- 5. GRUPO DE TVs GERAL
        WHEN GP.NomeGrupoPai IN ('TV', 'TVS') THEN 
            CASE 
                WHEN FI.CodProduto IN (66522, 56549, 66523, 55836, 64018, 66524, 56554, 64019, 71834, 66525, 66526, 71828, 58481, 70119, 63945, 70118, 71215, 68043, 68041, 68040, 71830, 69674, 69669, 69277, 77217, 77736) THEN 'SERRA/ES'
                WHEN PR.Marca = 'HQ' THEN 'MANAUS/AM' 
                ELSE 'TERCEIRIZADOS' 
            END
            
        -- 6. ELETROPORTÁTEIS, CLIMATIZAÇÃO, GELO E UTENSÍLIOS (HQ/3GREEN em Serra/ES, outros Terceirizados)
        WHEN GP.NomeGrupoPai IN (
            'AR CONDICIONADO', 'FRIGOBAR', 'FREEZER', 'REFRIGERADOR', 'GRILL E SANDUICHEIRAS', 
            'NOTEBOOK', 'FRITADEIRA', 'COOKTOPS', 'LAVADOURA LOUCAS', 'CERVEJEIRA', 
            'MAQUINA DE GELO', 'PANELA ELETRICA', 'SPLIT', 'ADEGA',
            'ACESSIBILIDADE E MOBILIDADE', 'BATEDEIRAS', 'CAFETEIRAS', 'CAIXA AMPLIFICADA', 'CAMERAS DIGITAIS', 
            'CELULAR', 'CHALEIRA', 'CLIMATIZADOR', 'COIFA', 'CONSOLE', 'COPOS E CANECAS', 'COZINHA', 
            'COZINHA CRIATIVA', 'CUBA TANQUE E PIA', 'DECORACAO', 'DEPURADOR', 'EMBALAGENS', 'ESCADA', 
            'ESCRITORIO', 'ESTANTE', 'EXPOSITOR', 'FERRAMENTAS', 'FERRAMENTAS PARA VEICULOS', 'FERRO A VAPOR', 
            'FILTROS', 'FOGAO', 'KIT DE FERRAMENTAS', 'LAVADORA DE ALTA PRESSAO', 'LAVADORA DE ROUPAS', 
            'LIQUIDIFICADOR', 'MALAS DE VIAGEM', 'MICROONDAS', 'MIXER', 'MULTIPROCESSADOR DE ALIMENTOS', 
            'PARTES E PECAS', 'PECAS PARA MOVEIS', 'PECAS SOLAR', 'PIPOQUEIRA ELETRICA', 'PNEUS', 
            'PURIFICADOR', 'REFRESQUEIRA', 'SECADORA', 'SISTEMAS DE SOM', 'TELEFONIA FIXA', 'TORNEIRAS', 
            'VAPORIZADOR', 'VENTILADOR', 'CUBA, TANQUE E PIA'
        ) THEN 
            CASE 
                WHEN PR.Marca IN ('HQ', '3GREEN') THEN 'SERRA/ES' 
                ELSE 'TERCEIRIZADOS' 
            END
            
        -- 7. RECOBERTA DE MARCAS TERCEIRIZADAS (Se não for fabricação própria homologada, joga em Terceirizados)
        WHEN PR.Marca NOT IN ('HQ', 'CORPC', '3GREEN', 'SKILL', 'EASYPC', 'QUANTUM') THEN 'TERCEIRIZADOS'
        
        -- 8. REGRA DEFAULT DE SEGURANÇA (O que sobrar e for marca própria, vai para Serra/ES)
        ELSE 'SERRA/ES'
    END AS Planta,
    
    -- Devolução inverte o sinal do estoque e do financeiro
    (FI.QtdNegociada * -1) AS quantidade_final,
    (FI.ValorTotal * -1) AS valor_final

FROM gold.sankhya.fato_operacoes FO
INNER JOIN itens_unicos FI ON FO.NumUnicoNota = FI.NumUnicoNota
INNER JOIN gold.sankhya.dim_produtos PR ON FI.CodProduto = PR.CodProduto
INNER JOIN gold.sankhya.dim_grupo_produtos GP ON PR.CodGrupoProduto = GP.CodGrupoProduto
LEFT JOIN gold.sankhya.dim_parceiros P ON FO.CodParceiro = P.CodParceiro
LEFT JOIN gold.sankhya.dim_vendedor V ON FO.CodVendedor = V.CodVendedor

WHERE 
    FO.DataNegociacao BETWEEN '2025-01-01' AND CURRENT_DATE()
    AND FO.TipoMovimento IN ('D-Devolução de venda')
    
    -- RANGE TOTAL DE TOPS DE ENTRADA/DEVOLUÇÃO (IMAGEM + ADICIONAIS HISTÓRICOS)
    AND FO.CodTipoOperacao IN (
        2220, 2201, 2234, 2227, 2284, 2252, 2285, 2210, 2215, 2221, 2200, 2233, 2249, 2259, 2230,
        2290, 2283
    )
    
    -- Filtros de consistência de cadastro de produtos
    AND GP.NomeGrupoPai <> 'COMPONENTES'
    AND GP.NomeGrupoPai <> 'PARTES E PECAS'
    
    -- Filtros comerciais (Gerentes/Lojas parceiras) — UPPER para ignorar case
    AND (
        UPPER(FO.NomeGerente) IN (
            'MERCADO LIVRE', 'MAGAZINE LUIZA', 'SHOPEE', 'VIA VAREJO', 'AMAZON', 
            'KABUM', 'MKTPLC MARTINS', 'B2W', 'WEBCONTINENTAL', 'CARREFOUR', 
            'TIKTOK SHOP', 'LEROY MERLIN', 'ZEMA', 'MADEIRA', 'EFÁCIL', 
            'IMPÉRIO', 'MAGENTO', 'INTER MARKETPLACE', 'LOJA BELMICRO SHOP',
            'LICITAÇÕES', 'VENDAS DIRETAS', 'VENDAS INTERNAS (BELMICRO)'
        )
        OR UPPER(FO.NomeGerente) LIKE '%LOJA OFICIAL%'
        OR UPPER(FO.NomeGerente) LIKE '%VENDAS EXTERNAS%'
    )
    
    -- Filtro de marcas e regras fiscais elegíveis
    AND (
        (PR.UsadoComo IN ('Venda (fabricação própria)', 'Revenda') AND GP.LinhaDeNegocio IN ('WordPC/Skill', 'Comprebel') AND PR.Marca IN ('HQ', '3GREEN', 'EASYPC', 'SKILL', 'QUANTUM', 'CORPC', 'FOXPC', 'AMD'))
        OR 
        (PR.UsadoComo IN ('Revenda', 'Venda (fabricação própria)') AND PR.Marca IN ('HQ', 'KONKA', '3GREEN') AND GP.NomeGrupoPai IN ('AR CONDICIONADO', 'FRIGOBAR', 'FORNO', 'NOTEBOOK', 'FRITADEIRA', 'REFRIGERADOR', 'GRILL E SANDUICHEIRAS', 'FREEZER', 'ADEGA', 'COOKTOPS', 'LAVADOURA LOUCAS', 'CERVEJEIRA', 'MAQUINA DE GELO', 'PANELA ELETRICA', 'MONITORES', 'TV', 'MONITOR') AND NOT (GP.LinhaDeNegocio IN ('WordPC/Skill', 'Comprebel') AND PR.Marca IN ('HQ', '3GREEN', 'EASYPC', 'SKILL', 'QUANTUM', 'CORPC', 'FOXPC', 'AMD')))
        OR 
        (PR.UsadoComo = 'Revenda' AND PR.Marca NOT IN ('HQ', '3GREEN', 'EASYPC', 'SKILL', 'QUANTUM', 'CORPC', 'FOXPC', 'AMD', 'KONKA'))
    )
ORDER BY FO.DataNegociacao DESC;